# Backtesting - Validação da Estratégia XGBoost

## Objetivo
Simular trades usando o modelo XGBoost treinado em dados históricos para validar a rentabilidade e viabilidade da estratégia antes de colocá-la em produção.

## O que será testado
* **Modelo:** XGBoost com 52 features multi-timeframe (1h, 15m, 4h)
* **Dataset:** `crypto_features_with_adaptive_targets` (~314k registros, 8 pares)
* **Features:** 13 features por timeframe + 13 de correlação BTC/ETH
* **Estratégia:** Compra quando probabilidade >= threshold, venda quando TP/SL atingido
* **Gestão de risco:** Stop Loss e Take Profit adaptativos baseados em ATR (1:2 risk/reward)
* **Custos:** Taxas de exchange (0.1%) e slippage (0.05%)

## Targets Adaptativos
* **TP:** close + 2.0 × ATR (adaptado à volatilidade)
* **SL:** close - 1.0 × ATR (controle de risco)
* **Risk/Reward:** 1:2 ratio fixo
* **Vantagens:** Adapta-se automaticamente a diferentes ativos e condições de mercado

## Métricas de Performance
* **Retorno total** vs Buy & Hold
* **Sharpe Ratio** - Retorno ajustado ao risco
* **Maximum Drawdown** - Maior queda do capital
* **Win Rate** - % de trades lucrativos
* **Profit Factor** - Lucro bruto / Prejuízo bruto
* **Número de trades** e tempo médio de holding

## Estrutura
1. Carregamento do modelo e dados
2. Configuração de parâmetros
3. Engine de backtesting
4. Execução em múltiplos thresholds
5. Análise de performance
6. Visualizações
7. Conclusões e recomendações

In [0]:
%pip install xgboost scikit-learn matplotlib seaborn

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import pyspark.sql.functions as f
from pyspark.sql.window import Window

# config visualizações
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)

print(f"pandas {pd.__version__}")
print(f"numPy {np.__version__}")
print(f"XGBoost {xgb.__version__}")

## Carregamento do Modelo e Dados
1. **Modelo XGBoost** treinado (arquivo .pkl)
2. **Dataset** com features e targets adaptativos
3. **Metadados** do treinamento (feature columns, parâmetros)

In [0]:
import joblib
import json
import os

model_path = os.path.join(os.path.dirname(os.getcwd()), 'models', 'xgboost_atr_target.pkl')
metadata_path = os.path.join(os.path.dirname(os.getcwd()), 'models', 'xgboost_atr_target_metadata.json')

# carrega modelo
if os.path.exists(model_path):
    model = joblib.load(model_path)
    print(f"Modelo carregado: {model_path}")
    print(f"   • Tipo: {type(model).__name__}")
    print(f"   • N_estimators: {model.n_estimators}")
    print(f"   • Max_depth: {model.max_depth}")
else:
    raise FileNotFoundError(f"Modelo não encontrado em {model_path}")

# carrega metadata (se existir)
if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    print(f"\nMetadata carregado:")
    print(f"   • Data treinamento: {metadata.get('train_date', 'N/A')}")
    print(f"   • AUC-ROC: {metadata.get('auc_roc', 'N/A')}")
    print(f"   • Features: {len(metadata.get('feature_columns', []))}")
    feature_cols = metadata.get('feature_columns', [])
else:
    print("\nMetadata não encontrado, usando features padrão")
    feature_cols = None

In [0]:
# Carregar dataset com features e targets adaptativos
import pyspark.sql.functions as F
from pyspark.sql.window import Window

df_spark = spark.table("workspace.default.crypto_features_with_adaptive_targets")

print(f"Dataset carregado:")
print(f"  • Registros: {df_spark.count():,}")
print(f"  • Colunas: {len(df_spark.columns)}")

# Período
period = df_spark.agg(F.min('timestamp'), F.max('timestamp')).collect()[0]
print(f"  • Período: {period[0]} até {period[1]}")
if period[0] and period[1]:
    print(f"  • Duração: {(period[1] - period[0]).days} dias")

# Estatísticas dos pares
print(f"\nPares disponíveis:")
for row in df_spark.groupBy('symbol').count().orderBy('symbol').collect():
    print(f"  • {row['symbol']}: {row['count']:,} registros")

# Converter para pandas para backtesting
print(f"\nConvertendo para pandas...")
df = df_spark.toPandas()

print(f"\nDataset pandas:")
print(f"  • Shape: {df.shape}")
print(f"  • Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Verificar targets adaptativos
if 'target_binary' in df.columns:
    target_ratio = df['target_binary'].mean()
    print(f"\nTargets adaptativos:")
    print(f"  • Ratio de targets positivos: {target_ratio:.2%}")
    print(f"  • TP/SL thresholds: Baseados em ATR (adaptativos por ativo)")

---

## Preparação das Features

Extrair as mesmas colunas usadas no treinamento e preparar o dataset para backtesting

In [0]:
# O modelo foi treinado com 96 features multi-timeframe (Lote 1)
feature_cols = [
    # Features 1h (timeframe principal operacional) - 28 features
    'log_return_1h', 'rsi_1h', 'volatility_1h', 'ma_close_1h', 'ma_volume_1h',
    'z_score_close_1h', 'z_score_volume_1h', 'atr_1h', 'momentum_1h', 'vol_change_1h',
    'bb_upper_1h', 'bb_lower_1h', 'bb_position_1h',
    # Lote 1 - 1h
    'vwap_1h', 'vwap_distance_1h', 'body_range_ratio_1h', 
    'upper_shadow_ratio_1h', 'lower_shadow_ratio_1h',
    'roc_5_1h', 'roc_10_1h', 'roc_20_1h', 'obv_1h', 'adx_1h', 'vol_ratio_1h',
    'resistance_break_1h', 'support_break_1h', 'hour_sin_1h', 'hour_cos_1h',
    
    # Features 15m (microestrutura de mercado) - 28 features
    'log_return_15m', 'rsi_15m', 'volatility_15m', 'ma_close_15m', 'ma_volume_15m',
    'z_score_close_15m', 'z_score_volume_15m', 'atr_15m', 'momentum_15m', 'vol_change_15m',
    'bb_upper_15m', 'bb_lower_15m', 'bb_position_15m',
    # Lote 1 - 15m
    'vwap_15m', 'vwap_distance_15m', 'body_range_ratio_15m',
    'upper_shadow_ratio_15m', 'lower_shadow_ratio_15m',
    'roc_5_15m', 'roc_10_15m', 'roc_20_15m', 'obv_15m', 'adx_15m', 'vol_ratio_15m',
    'resistance_break_15m', 'support_break_15m', 'hour_sin_15m', 'hour_cos_15m',
    
    # Features 4h (tendências de médio prazo) - 26 features
    'log_return_4h', 'rsi_4h', 'volatility_4h', 'ma_close_4h', 'ma_volume_4h',
    'z_score_close_4h', 'z_score_volume_4h', 'atr_4h', 'momentum_4h', 'vol_change_4h',
    'bb_upper_4h', 'bb_lower_4h', 'bb_position_4h',
    # Lote 1 - 4h
    'vwap_4h', 'vwap_distance_4h', 'body_range_ratio_4h',
    'upper_shadow_ratio_4h', 'lower_shadow_ratio_4h',
    'roc_5_4h', 'roc_10_4h', 'roc_20_4h', 'obv_4h', 'adx_4h', 'vol_ratio_4h',
    'resistance_break_4h', 'support_break_4h',
    
    # Features de correlação BTC/ETH - 13 features
    'btc_close', 'btc_log_return', 'btc_volume', 'btc_volatility',
    'eth_close', 'eth_log_return', 'eth_volume', 'eth_volatility',
    'relative_strength_btc', 'relative_strength_eth', 'volatility_ratio_btc',
    'volume_ratio_btc', 'beta_btc',
    
    # Lote 1 - Market Breadth - 1 feature
    'market_breadth'
]

print(f"Usando {len(feature_cols)} features multi-timeframe (Lote 1)")
print(f"\nGrupos de features:")
print(f"  • 1h (operacional): 28 features (13 originais + 15 Lote 1)")
print(f"  • 15m (microestrutura): 28 features (13 originais + 15 Lote 1)")
print(f"  • 4h (tendência): 26 features (13 originais + 13 Lote 1)")
print(f"  • Correlações BTC/ETH: 13 features")
print(f"  • Market Breadth: 1 feature")
print(f"  • TOTAL: {len(feature_cols)} features")

# Verificar se as colunas existem no dataset
missing_cols = [col for col in feature_cols if col not in df.columns]

if missing_cols:
    print(f"\nERRO: Features faltando no dataset: {missing_cols}")
    raise ValueError(f"Dataset não contém as features esperadas pelo modelo: {missing_cols}")
else:
    print(f"\n✓ Todas as {len(feature_cols)} features estão disponíveis")

# Preparar dataset para backtesting
print(f"\nPreparando dataset...")
initial_rows = len(df)

# Colunas necessárias para backtesting
required_cols = feature_cols + [
    'timestamp', 'symbol', 'close', 'open', 'high', 'low',
    'target_binary', 'tp_threshold', 'sl_threshold'
]

# Selecionar colunas
df_backtest = df[required_cols].copy()

# Remover linhas com NaN em features críticas
df_backtest = df_backtest.dropna(subset=feature_cols)

final_rows = len(df_backtest)
pct_kept = final_rows / initial_rows * 100

print(f"  • Registros iniciais: {initial_rows:,}")
print(f"  • Registros após limpeza: {final_rows:,}")
print(f"  • Mantidos: {pct_kept:.1f}%")
print(f"  • Símbolos: {df_backtest['symbol'].nunique()}")
print(f"  • Período: {df_backtest['timestamp'].min()} a {df_backtest['timestamp'].max()}")

print(f"\n✓ Dataset preparado com {len(feature_cols)} features do Lote 1")

## Configuração do Backtesting
Definir parâmetros da simulação:
* **Capital inicial:** $10,000
* **Tamanho de posição:** 95% do capital disponível
* **Taxas:** 0.1% por trade (maker/taker)
* **Slippage:** 0.05% (execução imperfeita)
* **Thresholds:** [0.55, 0.60, 0.65, 0.70] - múltiplos níveis de confiança
* **TP/SL:** Baseados em ATR (adaptativos por ativo)

In [0]:
# Parâmetros de capital e posição
INITIAL_CAPITAL = 10000.0  # USD
POSITION_SIZE = 0.95       # 95% do capital (5% reserva)

# Custos de trading
TRANSACTION_FEE = 0.001    # 0.1% (Binance spot)
SLIPPAGE = 0.0005          # 0.05% (deslizamento médio)
TOTAL_COST = TRANSACTION_FEE + SLIPPAGE  # 0.15% total por trade

# Thresholds de probabilidade para testar
THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70]

print(f"Capital e Posição:")
print(f"  • Capital inicial: ${INITIAL_CAPITAL:,.2f}")
print(f"  • Tamanho de posição: {POSITION_SIZE*100:.1f}%")
print(f"  • Capital por trade: ${INITIAL_CAPITAL * POSITION_SIZE:,.2f}")

print(f"\nCustos de Trading:")
print(f"  • Taxa de transação: {TRANSACTION_FEE*100:.2f}%")
print(f"  • Slippage estimado: {SLIPPAGE*100:.2f}%")
print(f"  • Custo total por trade: {TOTAL_COST*100:.2f}%")

print(f"\nThresholds de probabilidade: {THRESHOLDS}")

print(f"\nGestão de Risco:")
print(f"  • TP/SL: Adaptativos baseados em ATR")
print(f"  • TP = close + 2.0 × ATR")
print(f"  • SL = close - 1.0 × ATR")
print(f"  • Risk/Reward ratio: 1:2")
print(f"  • Varia por ativo e momento (adaptativo à volatilidade)")

print("\n✓ Configuração completa")

## Engine de Backtesting

Simulador de trades:
1. **Gera predições** do modelo para todo o dataset
2. **Abre posição** quando probabilidade >= threshold
3. **Fecha posição** quando:
   - Take Profit atingido
   - Stop Loss atingido
   - Probabilidade cai abaixo do threshold
4. **Registra** cada trade: entry, exit, P&L, motivo
5. **Calcula** equity curve e métricas

In [0]:
def run_backtest_single_symbol(df_symbol, threshold, initial_capital, position_size):
    """
    Backtest para UM ÚNICO símbolo com CAPITAL FIXO por trade.
    
    FIX: Usa tamanho de posição FIXO (sem compounding) para evitar crescimento exponencial.
    Cada trade usa o mesmo capital inicial, P&L é acumulado separadamente.
    """
    # Capital fixo por trade
    fixed_position_value = initial_capital * position_size
    
    # Acumulador de P&L
    cumulative_pnl = 0.0
    
    in_position = False
    position = {}
    trades = []
    equity_history = []
    
    # Iterar por índice numérico
    for i in range(len(df_symbol)):
        row = df_symbol.iloc[i]
        
        timestamp = row['timestamp']
        close = float(row['close'])
        high = float(row['high'])
        low = float(row['low'])
        prob = float(row['prob_alta'])
        symbol = str(row['symbol'])
        
        # Se em posição, verificar saída
        if in_position:
            exit_signal = False
            exit_reason = None
            exit_price = None
            
            # TP
            if high >= position['tp_price']:
                exit_signal, exit_reason = True, 'TP'
                exit_price = position['tp_price'] * (1 - TOTAL_COST)
            # SL
            elif low <= position['sl_price']:
                exit_signal, exit_reason = True, 'SL'
                exit_price = position['sl_price'] * (1 - TOTAL_COST)
            # Sinal inverteu
            elif prob < threshold:
                exit_signal, exit_reason = True, 'Signal'
                exit_price = close * (1 - TOTAL_COST)
            
            if exit_signal:
                # P&L deste trade
                profit_pct = ((exit_price / position['entry_price']) - 1) * 100
                profit_usd = ((exit_price / position['entry_price']) - 1) * fixed_position_value
                
                # Acumular P&L
                cumulative_pnl += profit_usd
                
                # Registrar trade
                trades.append({
                    'entry_time': position['entry_time'],
                    'exit_time': timestamp,
                    'symbol': symbol,
                    'entry_price': position['entry_price'],
                    'exit_price': exit_price,
                    'tp_price': position['tp_price'],
                    'sl_price': position['sl_price'],
                    'profit_pct': profit_pct,
                    'profit_usd': profit_usd,
                    'holding_hours': (timestamp - position['entry_time']).total_seconds() / 3600,
                    'exit_reason': exit_reason,
                    'entry_prob': position['entry_prob']
                })
                
                in_position = False
                position = {}
        
        # Se não em posição, verificar entrada
        elif prob >= threshold:
            entry_price = close * (1 + TOTAL_COST)
            
            # TP/SL do dataset
            if pd.notna(row['tp_threshold']) and pd.notna(row['sl_threshold']):
                tp_price = float(row['tp_threshold'])
                sl_price = float(row['sl_threshold'])
            else:
                # Fallback
                tp_price = entry_price * 1.02
                sl_price = entry_price * 0.99
            
            position = {
                'entry_time': timestamp,
                'entry_price': entry_price,
                'tp_price': tp_price,
                'sl_price': sl_price,
                'entry_prob': prob
            }
            in_position = True
        
        # Equity = capital inicial + P&L acumulado + P&L não realizado da posição aberta
        if in_position:
            # P&L não realizado da posição atual
            unrealized_pnl = ((close / position['entry_price']) - 1) * fixed_position_value
            total_equity = initial_capital + cumulative_pnl + unrealized_pnl
        else:
            total_equity = initial_capital + cumulative_pnl
        
        equity_history.append({
            'timestamp': timestamp,
            'equity': total_equity
        })
    
    return trades, equity_history


def run_backtest(df, threshold, initial_capital=INITIAL_CAPITAL, position_size=POSITION_SIZE):
    """
    Backtest multi-símbolo usando função isolada para cada símbolo.
    """
    all_trades = []
    symbol_equities = {}
    
    symbols = sorted(df['symbol'].unique())
    capital_per_symbol = initial_capital / len(symbols)
    
    print(f"\nProcessando {len(symbols)} símbolos com ${capital_per_symbol:,.0f} cada...")
    print(f"Tamanho de posição FIXO: ${capital_per_symbol * position_size:,.2f} por trade")
    
    # Processar cada símbolo isoladamente
    for sym in symbols:
        # Filtrar e copiar para garantir isolamento
        df_sym = df[df['symbol'] == sym].copy().sort_values('timestamp').reset_index(drop=True)
        
        # Backtest para este símbolo
        trades, equity = run_backtest_single_symbol(
            df_sym, threshold, capital_per_symbol, position_size
        )
        
        all_trades.extend(trades)
        symbol_equities[sym] = pd.DataFrame(equity)
    
    print(f"✓ Trades executados: {len(all_trades)}")
    
    # Consolidar trades
    trades_df = pd.DataFrame(all_trades)
    
    # Agregar equity
    all_timestamps = pd.concat([symbol_equities[sym]['timestamp'] for sym in symbols]).unique()
    all_timestamps = sorted(all_timestamps)
    
    equity_total = pd.DataFrame({'timestamp': all_timestamps})
    
    for sym in symbols:
        sym_equity = symbol_equities[sym].rename(columns={'equity': f'equity_{sym}'})
        equity_total = equity_total.merge(sym_equity, on='timestamp', how='left')
        equity_total[f'equity_{sym}'] = equity_total[f'equity_{sym}'].ffill().bfill().fillna(capital_per_symbol)
    
    # Somar equities
    equity_cols = [f'equity_{sym}' for sym in symbols]
    equity_total['equity'] = equity_total[equity_cols].sum(axis=1)
    
    equity_curve = equity_total[['timestamp', 'equity']].copy()
    
    print(f"✓ Equity curve: {len(equity_curve)} pontos")
    
    return trades_df, equity_curve

print("✓ Função de backtesting REESCRITA (v5 - CAPITAL FIXO)")
print("  • Usa TAMANHO DE POSIÇÃO FIXO (sem compounding)")
print("  • Cada trade usa o mesmo capital inicial")
print("  • P&L é acumulado separadamente")
print("  • Equity = capital inicial + P&L acumulado")
print("  • Elimina crescimento exponencial descontrolado")

### Bug Crítico Corrigido: Compounding Exponencial

**Problema identificado:**
* Backtest inicial gerava retornos absurdos (4+ bilhões %)
* Capital de SOL crescia de $1,250 → $10.8 trilhões em 1,909 trades  
* Trade individual usava 28.7 BILHÕES de unidades SOL ($1.3 trilhão)
* Profit_usd chegava a $1.4 bilhões em um único trade de 18%

**Causa raiz:**
* Compounding exponencial: reinvestimento de 100% dos lucros a cada trade
* Após milhares de trades, pequenos lucros compostos → crescimento astronômico
* Matematicamente correto, mas completamente irrealista (sem liquidez, slippage infinito)

**Solução implementada:**
* **Capital fixo por trade**: $1,187.50 (95% de $1,250 por símbolo)
* Cada trade usa sempre o mesmo tamanho inicial
* P&L acumulado separadamente: Equity = Capital Inicial + P&L Acumulado
* Elimina crescimento exponencial, mantém realismo

**Resultados pós-correção:**
* Threshold 0.55: 1,143% retorno em 5 anos (vs 4 bilhões %)
* Win rate: 70.6%, Profit médio: 0.68%
* Valores realistas e verificados

In [0]:
print("\nGerando predições do modelo...\n")

# Extrair features como array
X = df_backtest[feature_cols].values

print(f"Shape das features: {X.shape}")

# Gerar probabilidades
print("Calculando probabilidades...")
y_proba = model.predict_proba(X)[:, 1]  # Probabilidade da classe 1 (alta)

# Adicionar ao dataframe
df_backtest['prob_alta'] = y_proba

print(f"\nPredições geradas:")
print(f"  • Mín: {y_proba.min():.4f}")
print(f"  • Máx: {y_proba.max():.4f}")
print(f"  • Média: {y_proba.mean():.4f}")
print(f"  • Mediana: {np.median(y_proba):.4f}")

# Distribuição das probabilidades
print(f"\nDistribuição por threshold:")
for threshold in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75]:
    count = (y_proba >= threshold).sum()
    pct = count / len(y_proba) * 100
    print(f"  • >= {threshold:.2f}: {count:,} registros ({pct:.2f}%)")

print(f"\n✓ Predições prontas para backtesting")

In [0]:
print("DEBUG: Teste de isolamento para BNB\n")
print("=" * 60)

# Filtrar apenas BNB
df_bnb_test = df_backtest[df_backtest['symbol'] == 'BNB/USDT'].copy()
df_bnb_test = df_bnb_test.sort_values('timestamp').reset_index(drop=True)

print(f"Dataset BNB: {len(df_bnb_test)} linhas")
print(f"Símbolos únicos: {df_bnb_test['symbol'].unique()}")

# Verificar linha problemática
target_ts = pd.Timestamp('2021-04-02 03:00:00')
test_row_idx = df_bnb_test[df_bnb_test['timestamp'] == target_ts].index

if len(test_row_idx) > 0:
    test_row_idx = test_row_idx[0]
    print(f"\nLinha alvo no índice: {test_row_idx}")
    
    # Verificar dados ANTES de entrar na função
    row_before = df_bnb_test.iloc[test_row_idx]
    print(f"\nDADOS ANTES DA FUNÇÃO:")
    print(f"  iloc[{test_row_idx}] = Timestamp: {row_before['timestamp']}")
    print(f"  iloc[{test_row_idx}] = Symbol: {row_before['symbol']}")
    print(f"  iloc[{test_row_idx}] = Close: {row_before['close']}")
    print(f"  iloc[{test_row_idx}] = Prob: {row_before['prob_alta']:.4f}")
    
    # SIMULAR A ITERAÇÃO DA FUNÇÃO
    print(f"\n\nSIMULANDO ITERAÇÃO DA FUNÇÃO:")
    print("=" * 60)
    
    # Simular exatamente o que a função faz
    threshold = 0.50
    capital = INITIAL_CAPITAL / 8  # Divide por 8 símbolos
    position_size = POSITION_SIZE
    
    in_position = False
    
    # Iterar até a linha problemática + algumas depois
    for i in range(max(0, test_row_idx - 2), min(len(df_bnb_test), test_row_idx + 3)):
        row = df_bnb_test.iloc[i]
        
        timestamp = row['timestamp']
        close = float(row['close'])
        prob = float(row['prob_alta'])
        symbol = str(row['symbol'])
        
        print(f"\n[{i}] {timestamp}:")
        print(f"  Symbol: {symbol}")
        print(f"  Close: {close}")
        print(f"  Prob: {prob:.4f}")
        
        # Check entrada
        if prob >= threshold and not in_position:
            entry_price = close * (1 + TOTAL_COST)
            in_position = True
            print(f"  → ENTRADA: ${entry_price:.2f}")
        
        # Check saída
        elif prob < threshold and in_position:
            exit_price = close * (1 - TOTAL_COST)
            print(f"  → SAÍDA (Signal): ${exit_price:.2f}")
            in_position = False
            
            if i == test_row_idx:
                print(f"\n  *** ESTE É O TRADE PROBLEMÁTICO ***")
                print(f"  *** Close usado: {close} ***")
                print(f"  *** Exit price: {exit_price:.2f} ***")
                if exit_price > 10000:
                    print(f"\n  ❌ BUG REPRODUZIDO!")
                else:
                    print(f"\n  ✅ Valor correto!")
else:
    print(f"\n❌ Timestamp {target_ts} não encontrado no dataset BNB")

## Execução dos Backtests

Testar a estratégia com múltiplos thresholds de probabilidade

In [0]:
backtest_results = {}

for threshold in THRESHOLDS:
    print(f"\nThreshold: {threshold:.2f}")
    print("-" * 50)
    
    # Executar backtest
    trades_df, equity_curve = run_backtest(df_backtest, threshold)
    
    # Estatísticas básicas
    if len(trades_df) > 0:
        total_return = (equity_curve['equity'].iloc[-1] / INITIAL_CAPITAL - 1) * 100
        n_trades = len(trades_df)
        win_rate = (trades_df['profit_pct'] > 0).sum() / n_trades * 100
        avg_profit = trades_df['profit_pct'].mean()
        
        print(f"  • Trades executados: {n_trades}")
        print(f"  • Retorno total: {total_return:.2f}%")
        print(f"  • Win rate: {win_rate:.1f}%")
        print(f"  • Profit médio: {avg_profit:.2f}%")
    else:
        print(f"  • Nenhum trade executado")
    
    # Armazenar resultados
    backtest_results[threshold] = {
        'trades': trades_df,
        'equity': equity_curve
    }

---

## Métricas de Performance

Calcular métricas detalhadas para cada threshold:
* **Retorno total** - Performance absoluta
* **Sharpe Ratio** - Retorno ajustado ao risco
* **Maximum Drawdown** - Maior perda acumulada
* **Win Rate** - Proporção de trades lucrativos
* **Profit Factor** - Razão lucro/prejuízo
* **Average Profit/Loss** - P&L médio por trade

In [0]:
def calculate_metrics(trades_df, equity_curve, initial_capital=INITIAL_CAPITAL):
    """
    Calcula métricas de performance do backtest.
    """
    if len(trades_df) == 0:
        return {
            'total_return_pct': 0,
            'sharpe_ratio': 0,
            'max_drawdown_pct': 0,
            'win_rate_pct': 0,
            'profit_factor': 0,
            'n_trades': 0,
            'avg_profit_pct': 0,
            'avg_holding_hours': 0,
            'final_capital': initial_capital
        }
    
    # 1. Retorno total
    final_capital = equity_curve['equity'].iloc[-1]
    total_return_pct = (final_capital / initial_capital - 1) * 100
    
    # 2. Sharpe Ratio (anualizado)
    # Calcular retornos diários da equity curve
    equity_curve['returns'] = equity_curve['equity'].pct_change()
    daily_returns = equity_curve.groupby(equity_curve['timestamp'].dt.date)['returns'].sum()
    
    if len(daily_returns) > 1 and daily_returns.std() > 0:
        sharpe_ratio = (daily_returns.mean() / daily_returns.std()) * np.sqrt(365)
    else:
        sharpe_ratio = 0
    
    # 3. Maximum Drawdown
    equity_curve['cummax'] = equity_curve['equity'].cummax()
    equity_curve['drawdown'] = (equity_curve['equity'] / equity_curve['cummax'] - 1) * 100
    max_drawdown_pct = equity_curve['drawdown'].min()
    
    # 4. Win Rate
    winning_trades = (trades_df['profit_pct'] > 0).sum()
    win_rate_pct = winning_trades / len(trades_df) * 100
    
    # 5. Profit Factor
    gross_profit = trades_df[trades_df['profit_pct'] > 0]['profit_usd'].sum()
    gross_loss = abs(trades_df[trades_df['profit_pct'] < 0]['profit_usd'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else 0
    
    # 6. Médias
    avg_profit_pct = trades_df['profit_pct'].mean()
    avg_holding_hours = trades_df['holding_hours'].mean()
    
    return {
        'total_return_pct': total_return_pct,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown_pct': max_drawdown_pct,
        'win_rate_pct': win_rate_pct,
        'profit_factor': profit_factor,
        'n_trades': len(trades_df),
        'avg_profit_pct': avg_profit_pct,
        'avg_holding_hours': avg_holding_hours,
        'final_capital': final_capital,
        'gross_profit': gross_profit,
        'gross_loss': gross_loss
    }

print("\nCalculando métricas de performance...\n")
print("="*70)

metrics_summary = []

for threshold in THRESHOLDS:
    trades_df = backtest_results[threshold]['trades']
    equity_curve = backtest_results[threshold]['equity']
    
    metrics = calculate_metrics(trades_df, equity_curve)
    metrics['threshold'] = threshold
    metrics_summary.append(metrics)
    
    print(f"\nThreshold: {threshold:.2f}")
    print("-" * 50)
    print(f"  Retorno Total: {metrics['total_return_pct']:.2f}%")
    print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
    print(f"  Max Drawdown: {metrics['max_drawdown_pct']:.2f}%")
    print(f"  Win Rate: {metrics['win_rate_pct']:.1f}%")
    print(f"  Profit Factor: {metrics['profit_factor']:.2f}")
    print(f"  Trades: {metrics['n_trades']}")
    print(f"  Holding médio: {metrics['avg_holding_hours']:.1f}h")
    print(f"  Capital final: ${metrics['final_capital']:,.2f}")

print("\n" + "="*70)

# df com todas as métricas
metrics_df = pd.DataFrame(metrics_summary)
print("\nMétricas calculadas para todos os thresholds")

## Comparação com Buy & Hold
Calcular o retorno de uma estratégia passiva (comprar e segurar) para benchmarking

In [0]:
print("\nCalculando retorno Buy & Hold...\n")

# Calcular retorno médio de todos os pares
buy_hold_returns = []

for symbol in df_backtest['symbol'].unique():
    df_symbol = df_backtest[df_backtest['symbol'] == symbol].copy()
    
    if len(df_symbol) > 0:
        first_price = df_symbol.iloc[0]['close']
        last_price = df_symbol.iloc[-1]['close']
        symbol_return = (last_price / first_price - 1) * 100
        
        buy_hold_returns.append({
            'symbol': symbol,
            'return_pct': symbol_return,
            'first_price': first_price,
            'last_price': last_price
        })

buy_hold_df = pd.DataFrame(buy_hold_returns)
avg_buy_hold_return = buy_hold_df['return_pct'].mean()

print(f"Retornos Buy & Hold por par:")
print("-" * 50)
for _, row in buy_hold_df.iterrows():
    print(f"  • {row['symbol']:15s}: {row['return_pct']:+7.2f}%")

print("-" * 50)
print(f"  Média: {avg_buy_hold_return:+.2f}%")

# Comparar com melhor threshold
best_threshold_idx = metrics_df['total_return_pct'].idxmax()
best_threshold = metrics_df.loc[best_threshold_idx, 'threshold']
best_return = metrics_df.loc[best_threshold_idx, 'total_return_pct']

print(f"\nComparação:")
print(f"  • Buy & Hold: {avg_buy_hold_return:+.2f}%")
print(f"  • Melhor estratégia (threshold={best_threshold:.2f}): {best_return:+.2f}%")
print(f"  • Diferença: {best_return - avg_buy_hold_return:+.2f}%")

if best_return > avg_buy_hold_return:
    print(f"\nA estratégia superou o Buy & Hold!")
else:
    print(f"\nA estratégia não superou o Buy & Hold")

## Visualizações de Performance

Gráficos para análise visual:
1. **Equity curves** - Evolução do capital
2. **Drawdown** - Quedas do capital
3. **Distribuição de retornos** - Histogram de P&L
4. **Win rate por threshold** - Comparação
5. **Sharpe ratio por threshold** - Qualidade ajustada ao risco

In [0]:
fig, ax = plt.subplots(figsize=(16, 7))

# Plot equity curve para cada threshold
for threshold in THRESHOLDS:
    equity_curve = backtest_results[threshold]['equity']
    if len(equity_curve) > 0:
        ax.plot(equity_curve['timestamp'], equity_curve['equity'], 
                label=f"Threshold {threshold:.2f}", linewidth=2, alpha=0.8)

# Linha de capital inicial
ax.axhline(y=INITIAL_CAPITAL, color='gray', linestyle='--', 
           label='Capital Inicial', linewidth=1, alpha=0.5)

ax.set_xlabel('Data', fontsize=12)
ax.set_ylabel('Capital ($)', fontsize=12)
ax.set_title('Equity Curves - Evolução do Capital por Threshold', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(16, 7))

# usando o melhor threshold
best_threshold_idx = metrics_df['total_return_pct'].idxmax()
best_threshold = metrics_df.loc[best_threshold_idx, 'threshold']

equity_curve = backtest_results[best_threshold]['equity'].copy()
equity_curve['cummax'] = equity_curve['equity'].cummax()
equity_curve['drawdown'] = (equity_curve['equity'] / equity_curve['cummax'] - 1) * 100

ax.fill_between(equity_curve['timestamp'], equity_curve['drawdown'], 0, 
                 color='red', alpha=0.3, label='Drawdown')
ax.plot(equity_curve['timestamp'], equity_curve['drawdown'], 
        color='darkred', linewidth=2, alpha=0.8)

max_dd = equity_curve['drawdown'].min()
max_dd_date = equity_curve.loc[equity_curve['drawdown'].idxmin(), 'timestamp']

ax.axhline(y=max_dd, color='red', linestyle='--', 
           label=f'Max Drawdown: {max_dd:.2f}%', linewidth=2, alpha=0.7)

ax.set_xlabel('Data', fontsize=12)
ax.set_ylabel('Drawdown (%)', fontsize=12)
ax.set_title(f'Drawdown ao Longo do Tempo (Threshold {best_threshold:.2f})', 
             fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max DD: {max_dd:.2f}% em {max_dd_date}")

In [0]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, threshold in enumerate(THRESHOLDS):
    trades_df = backtest_results[threshold]['trades']
    
    if len(trades_df) > 0:
        ax = axes[idx]
        
        # Histogram
        ax.hist(trades_df['profit_pct'], bins=30, color='steelblue', 
                alpha=0.7, edgecolor='black')
        
        # Linha vertical em zero
        ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
        
        # Estatísticas
        mean_profit = trades_df['profit_pct'].mean()
        median_profit = trades_df['profit_pct'].median()
        
        ax.axvline(x=mean_profit, color='green', linestyle='-', 
                   linewidth=2, alpha=0.7, label=f'Média: {mean_profit:.2f}%')
        ax.axvline(x=median_profit, color='orange', linestyle='-', 
                   linewidth=2, alpha=0.7, label=f'Mediana: {median_profit:.2f}%')
        
        ax.set_xlabel('Retorno por Trade (%)', fontsize=10)
        ax.set_ylabel('Frequência', fontsize=10)
        ax.set_title(f'Threshold {threshold:.2f} - {len(trades_df)} trades', 
                     fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        axes[idx].text(0.5, 0.5, 'Sem trades', 
                       ha='center', va='center', fontsize=14)
        axes[idx].set_xticks([])
        axes[idx].set_yticks([])

# Esconder último subplot vazio
if len(THRESHOLDS) < 6:
    axes[-1].axis('off')

plt.tight_layout()
plt.show()

print(f"\nDistribuições de retorno geradas para {len(THRESHOLDS)} thresholds")

In [0]:
print("\nGerando comparação de métricas...\n")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Win Rate
ax = axes[0]
colors = ['green' if x > 50 else 'red' for x in metrics_df['win_rate_pct']]
ax.bar(metrics_df['threshold'].astype(str), metrics_df['win_rate_pct'], 
       color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='50%')
ax.set_xlabel('Threshold', fontsize=11)
ax.set_ylabel('Win Rate (%)', fontsize=11)
ax.set_title('Win Rate por Threshold', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 2. Sharpe Ratio
ax = axes[1]
colors = ['green' if x > 0 else 'red' for x in metrics_df['sharpe_ratio']]
ax.bar(metrics_df['threshold'].astype(str), metrics_df['sharpe_ratio'], 
       color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Threshold', fontsize=11)
ax.set_ylabel('Sharpe Ratio', fontsize=11)
ax.set_title('Sharpe Ratio por Threshold', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 3. Total Return
ax = axes[2]
colors = ['green' if x > 0 else 'red' for x in metrics_df['total_return_pct']]
ax.bar(metrics_df['threshold'].astype(str), metrics_df['total_return_pct'], 
       color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(y=avg_buy_hold_return, color='blue', linestyle='--', 
           linewidth=2, alpha=0.7, label=f'Buy & Hold: {avg_buy_hold_return:.1f}%')
ax.set_xlabel('Threshold', fontsize=11)
ax.set_ylabel('Retorno Total (%)', fontsize=11)
ax.set_title('Retorno Total por Threshold', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Comparação de Métricas por Threshold', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Análise Detalhada de Trades
Investigar padrões em trades vencedores e perdedores

In [0]:
print("\nAnalisando trades do melhor threshold...\n")

# Usar melhor threshold
best_threshold_idx = metrics_df['total_return_pct'].idxmax()
best_threshold = metrics_df.loc[best_threshold_idx, 'threshold']
trades_df = backtest_results[best_threshold]['trades'].copy()

if len(trades_df) > 0:
    print(f"Threshold analisado: {best_threshold:.2f}")
    print(f"Total de trades: {len(trades_df)}\n")
    
    # Separar vencedores e perdedores
    winners = trades_df[trades_df['profit_pct'] > 0].copy()
    losers = trades_df[trades_df['profit_pct'] <= 0].copy()
    
    print("="*70)
    print("✅ TRADES VENCEDORES")
    print("="*70)
    print(f"Quantidade: {len(winners)} ({len(winners)/len(trades_df)*100:.1f}%)\n")
    
    if len(winners) > 0:
        print(f"Estatísticas:")
        print(f"  • Profit médio: {winners['profit_pct'].mean():.2f}%")
        print(f"  • Profit mediano: {winners['profit_pct'].median():.2f}%")
        print(f"  • Maior profit: {winners['profit_pct'].max():.2f}%")
        print(f"  • Holding médio: {winners['holding_hours'].mean():.1f}h")
        print(f"  • Lucro total: ${winners['profit_usd'].sum():,.2f}")
        
        print(f"\nMotivos de saída:")
        exit_reasons = winners['exit_reason'].value_counts()
        for reason, count in exit_reasons.items():
            pct = count / len(winners) * 100
            print(f"  • {reason}: {count} ({pct:.1f}%)")
        
        print(f"\nTop 5 melhores trades:")
        print(winners.nlargest(5, 'profit_pct')[['entry_time', 'symbol', 'profit_pct', 
                                                   'holding_hours', 'exit_reason']].to_string(index=False))
    
    print("\n" + "="*70)
    print("❌ TRADES PERDEDORES")
    print("="*70)
    print(f"Quantidade: {len(losers)} ({len(losers)/len(trades_df)*100:.1f}%)\n")
    
    if len(losers) > 0:
        print(f"Estatísticas:")
        print(f"  • Loss médio: {losers['profit_pct'].mean():.2f}%")
        print(f"  • Loss mediano: {losers['profit_pct'].median():.2f}%")
        print(f"  • Maior loss: {losers['profit_pct'].min():.2f}%")
        print(f"  • Holding médio: {losers['holding_hours'].mean():.1f}h")
        print(f"  • Prejuízo total: ${losers['profit_usd'].sum():,.2f}")
        
        print(f"\nMotivos de saída:")
        exit_reasons = losers['exit_reason'].value_counts()
        for reason, count in exit_reasons.items():
            pct = count / len(losers) * 100
            print(f"  • {reason}: {count} ({pct:.1f}%)")
        
        print(f"\nTop 5 piores trades:")
        print(losers.nsmallest(5, 'profit_pct')[['entry_time', 'symbol', 'profit_pct', 
                                                   'holding_hours', 'exit_reason']].to_string(index=False))
    
    print("\n" + "="*70)
    
    # Análise por símbolo
    print("\nPERFORMANCE POR SÍMBOLO")
    print("="*70)
    symbol_stats = trades_df.groupby('symbol').agg({
        'profit_pct': ['count', 'mean', 'sum'],
        'profit_usd': 'sum'
    }).round(2)
    symbol_stats.columns = ['Trades', 'Avg_Profit%', 'Total_Return%', 'Total_USD']
    symbol_stats = symbol_stats.sort_values('Total_USD', ascending=False)
    print(symbol_stats.to_string())
    
else:
    print("Nenhum trade executado")

## Resumo e Recomendações

### Key Findings

O backtesting revelou insights importantes sobre a viabilidade da estratégia:

#### Performance Geral
* **Melhor threshold identificado** e suas métricas
* **Comparação com Buy & Hold** - A estratégia conseguiu superar?
* **Sharpe Ratio** - Qualidade do retorno ajustado ao risco
* **Maximum Drawdown** - Risco máximo observado

#### Padrões Identificados
* **Win rate** - Proporção de trades lucrativos
* **Profit factor** - Relação lucro/prejuízo
* **Exit reasons** - Principais motivos de saída (TP, SL, Signal)
* **Holding time** - Duração média das posições

### Recomendações para Melhoria

#### 1. Otimização de Parâmetros
* **Threshold ótimo:** Usar o threshold que maximizou Sharpe Ratio
* **TP/SL dinâmicos:** Ajustar baseado em volatilidade (ATR)
* **Position sizing:** Testar Kelly Criterion ou tamanhos variáveis

#### 2. Gestão de Risco
* **Limite diário de trades:** Evitar overtrading
* **Maximum drawdown stop:** Parar operações se DD > X%
* **Diversificação:** Ajustar exposição por par

#### 3. Melhorias no Modelo
* **Feature engineering:** Adicionar indicadores que faltaram
* **Ensemble:** Combinar múltiplos modelos
* **Retrain schedule:** Retreinar periodicamente (ex: mensal)

#### 4. Validação Adicional
* **Walk-forward analysis:** Testar em múltiplos períodos
* **Monte Carlo simulation:** Simular cenários alternativos
* **Stress test:** Performance em crashes de mercado

### Próximos Passos

1. **Ajustar parâmetros** baseado nos resultados do backtest
2. **Implementar melhorias** de gestão de risco
3. **Paper trading** - Testar em ambiente de simulação real
4. **Monitoramento contínuo** após deploy em produção